In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import scipy.stats as stats
from scipy.stats import spearmanr, hypergeom

from statsmodels.stats.multitest import multipletests

# GO enrichment
import gseapy as gp

## CellOracle
import celloracle as co

In [ ]:
## DATA
adata = sc.read_h5ad("../data/data_diff_express_lncRNA.h5ad")
base_GRN = pd.read_parquet('../data/celloracle_data/base_GRN_lncRNA_both_edge_list.parquet')

## Simulate shift

Also extracts expected expression shift after KO

In [ ]:
def evaluate_ko_impact_on_target(oracle_obj, target_gene, cluster_name=None):
    """
    Compara la expresión de un gen ANTES y DESPUÉS de la simulación de CellOracle.
    """
    print(f"--- ANALIZANDO IMPACTO DEL KO IN-SILICO SOBRE: {target_gene} ---")
    
    # 1. Encontrar el índice del gen en la matriz
    if target_gene not in oracle_obj.adata.var_names:
        print(f"Error: El gen {target_gene} no está en la red de CellOracle.")
        return
    
    gene_idx = oracle_obj.adata.var_names.get_loc(target_gene)
    
    # 2. Filtrar por cluster (Opcional, para ver el efecto en un estado específico)
    if cluster_name:
        # Asumiendo que tus clusters están en 'leiden' o 'phenotype_state'
        mask = oracle_obj.adata.obs['leiden'] == str(cluster_name)
    else:
        mask = np.ones(oracle_obj.adata.n_obs, dtype=bool) # Todas las células
        
    # 3. Extraer los Arrays (Microestados)
    # Expresión ORIGINAL (Estado Sano)
    expr_original = oracle_obj.adata.X[mask, gene_idx]
    
    # Expresión SIMULADA (Estado tras el KO)
    # (CellOracle suele guardar la simulación en una capa llamada 'simulated_count')
    expr_simulada = oracle_obj.adata.layers['simulated_count'][mask, gene_idx]
    
    # Si las matrices son dispersas, las aplanamos
    if hasattr(expr_original, "toarray"):
        expr_original = expr_original.toarray().flatten()
        expr_simulada = expr_simulada.toarray().flatten()

    # 4. Calcular el Valor Macroscópico (El Delta Termodinámico)
    media_original = np.mean(expr_original)
    media_simulada = np.mean(expr_simulada)
    delta_absoluto = media_simulada - media_original
    
    print(f"\nResultados para el cluster: {cluster_name if cluster_name else 'Global'}")
    print(f" -> Media Original (WT):  {media_original:.4f}")
    print(f" -> Media Simulada (KO): {media_simulada:.4f}")
    print(f" -> Shift Absoluto (Δ):  {delta_absoluto:.4f}")
    
    if media_original > 0:
        fold_change = media_simulada / media_original
        print(f" -> Fold Change:         {fold_change:.2f}x")
    
    # 5. Visualización de la Transición de Fase
    plt.figure(figsize=(6, 4))
    sns.kdeplot(expr_original, fill=True, color="blue", label="Original (WT)", alpha=0.5)
    sns.kdeplot(expr_simulada, fill=True, color="red", label="Simulado (KO)", alpha=0.5)
    plt.axvline(media_original, color='blue', linestyle='--')
    plt.axvline(media_simulada, color='red', linestyle='--')
    
    plt.title(f"Efecto In-Silico sobre {target_gene}")
    plt.xlabel("Nivel de Expresión (Log-Normalized)")
    plt.ylabel("Densidad de Células")
    plt.legend()
    plt.show()

# === EJEMPLO DE USO ===
# Supongamos que simulaste apagar Sox2 o tu TF proxy, 
# y quieres ver cómo predice el modelo que caerá Pax6 en el Ectodermo (Cluster 3)
evaluate_ko_impact_on_target(oracle, target_gene='Pax6', cluster_name='3')